Test file for typo generation

In [461]:
import copy
import math
import random
import string

In [2500]:
def read_iob2_file_wPOS(path):    
    """
    Read provided Universal NER iob2 file
    
    :param path: path to read from
    :returns: list with sequences of words, NER labels and POS tags for each sentence
    """
    data = []
    current_words = []
    current_ner_tags = []
    current_pos_tag = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()

        if line:
            if line[0] == '#':
                continue # skip comments
            tok = line.split(' ')
            #print(tok)
            current_words.append(tok[1])
            current_ner_tags.append(tok[2])
            current_pos_tag.append(tok[3])
        else:
            if current_words:  # skip empty lines
                data.append((current_words, current_ner_tags,current_pos_tag))
            current_words = []
            current_ner_tags = []
            current_pos_tag = []

    # check for last one
    if current_ner_tags != []:
        data.append((current_words, current_ner_tags,current_pos_tag))
    return data

numerical_expressions = ['zero', 'one', 'two', 'three', 'four',
                        'five', 'six', 'seven', 'eight', 'nine',
                        'ten', 'eleven', 'twelve', 'thirteen',
                        'fifteen', 'twenty', 'thirty', 'forty',
                        'fifty', 'hundred', 'thousand',
                        'million', 'billion'
                        '1', '2', '3', '4', '5', '6', '7', '8', '9', '0']
typo_types = ['insertion', 'replacement', 'deletion', 'transposition']
typo_type_samp_dist = [0.1525, 0.2825, 0.2825, 0.2825]

# First tuple are horizontally adjacent characters on keyboard
# Second tuple says whether a key is a left-handed key or not
keyboard_info = {
    'Q': (('W'), True),
    'W': (('Q','E'), True),
    'E': (('W','R'), True),
    'R': (('E','T'), True),
    'T': (('R','Y'), True),
    'Y': (('T','U'), False),
    'U': (('Y','I'), False),
    'I': (('U','O'), False),
    'O': (('I','P'), False),
    'P': (('O'), False),
    'A': (('S'), True),
    'S': (('A','D'), True),
    'D': (('S','F'), True),
    'F': (('D','G'), True),
    'G': (('F','H'), True),
    'H': (('G','J'), False),
    'J': (('H','K'), False),
    'K': (('J','L'), False),
    'L': (('K'), False),
    'Z': (('X'), True),
    'X': (('Z','C'), True),
    'C': (('X','V'), True),
    'V': (('C','B'), True),
    'B': (('V','N'), False),
    'N': (('B','M'), False),
    'M': (('N'), False),
}
# left_hand_keys = [i[0] for i in keyboard_info.items() if i[1][1] == True]
# right_hand_keys = [i[0] for i in keyboard_info.items() if i[1][1] == False]

def typo_generation(data, corruption_ratio=0.05, corruption_variation=1.0, min_typos=0, max_typos=100):
    """Injects typos into the provided data.
    Parameters
    ----------
    data : list 
        Pass the data returned from the parser function, should be a list ex. [([words],[NER tags],[POS tags]),.....]
    corruption_ratio : float
        Determines what percentage of the words are expected to have typos injected
    corruption_variation : float
        Determines how much the corruption ratio varies from sentence to sentence
    min_typos : int
        Limits how few typos there can be per sentence. In short sentences, corruption ratio may not be high enough at all to ensure a typo is created. This forces there to always be some typos per sentence.
    max_typos : int
        Limits how many typos there can be per sentence. With high enough corruption variation, there can be too many typos by chance.

    Returns
    -------
    typo_data : list 
        A list in the same format as the input, but, now with typos of a variety of forms, plus an additional list of ints appended to the end that count how many errors are in a particular word.
    Notes
    -----
    """
    typo_data = list(data)

    for data_point_i, data_point in enumerate(typo_data):
        sentence = data_point[0]
        if len(data_point) < 4: # debug
            data_point.append([0]*len(sentence))

        expected_typos = corruption_ratio * len(sentence)

        # Weight words based on square root of their length for typo sampling
        word_samp_dist = [math.sqrt(len(word)) for word in sentence]

        # Remove words that have any number-related components from sampling to avoid token corruption
        # Additionally, remove one-character-long words from being sampled
        for i in range(len(sentence)):
            for exp in numerical_expressions:
                if exp in sentence[i] or len(sentence[i]) < 2:
                    word_samp_dist[i] = 0

        # Normalize word sampling dist
        norm_word_samp_dist = [float(i)/sum(word_samp_dist) for i in word_samp_dist]

        # Typo injection loop
        typos_to_perform = int(round(expected_typos+corruption_variation*(random.random()-0.5)))
        typos_to_perform = max(typos_to_perform, min_typos)
        typos_to_perform = min(typos_to_perform, max_typos)

        for _ in range(typos_to_perform):
            print("~~~~~~~~~~~~~~~~~~~~~~~~~")
            # Sample a word from word sampling dist and add to error count
            sampled_word, sampled_word_i = list_sample(sentence, norm_word_samp_dist)
            data_point[3][sampled_word_i] += 1

            print(f"Sampled word: {sampled_word}")

            isAllOneHand = True
            transposition_set_zero = []
            for i in range(len(sampled_word)-1):
                cur = sampled_word[i]
                nxt = sampled_word[i+1]

                if not (cur in string.ascii_letters) or not (nxt in string.ascii_letters):
                    transposition_set_zero.append(i)
                    continue
                if i == 0 and cur.isupper() == nxt.islower():
                    transposition_set_zero.append(i)
                    continue
                a = keyboard_info.get(cur.upper())[1]
                b = keyboard_info.get(nxt.upper())[1]
                if a == None or b == None:
                    transposition_set_zero.append(i)
                elif a == b:
                    transposition_set_zero.append(i)
                else:
                    isAllOneHand = False

            print(f"transposition_set_zero: {transposition_set_zero}")

            # Sample a typo type
            if not isAllOneHand:
                sampled_typo_type, _ = list_sample(typo_types, typo_type_samp_dist)
                sampled_typo_type = "transposition"
            else:
                sampled_typo_type, _ = list_sample(typo_types[:3], [float(i)/sum(typo_type_samp_dist[:3]) for i in typo_type_samp_dist])
            
            print(f"Typo type: {sampled_typo_type}")

            # Create character sampling distribution
            char_samp_dist = [0]*len(sampled_word)
            char_samp_dist[1] = 0.1
            char_samp_dist[-1] = 0.2
            interp_indices = range(2,len(sampled_word)-1)
            for i in interp_indices:
                char_samp_dist[i] = 0.1+0.1*(i-1)/(len(interp_indices)+1)
            for i in range(len(sampled_word)):
                if not sampled_word[i] in string.ascii_letters:
                    char_samp_dist[i] = 0
                if sampled_typo_type == "transposition":
                    if not (i == 0 and sampled_word[i].isupper() == sampled_word[i+1].islower()):
                        char_samp_dist[0] = 0.05
                    for j in transposition_set_zero:
                        char_samp_dist[j] = 0
                    char_samp_dist[-1] = 0

            # Normalize character sampling distribution and sample
            norm_char_samp_dist = [float(i)/sum(char_samp_dist) for i in char_samp_dist]
            sampled_char, sampled_char_i = list_sample(sampled_word, norm_char_samp_dist)

            print(f"Character sampling distribution: {norm_char_samp_dist}")

            # Perform typo
            if sampled_typo_type == "insertion":
                # Check if the nearest ASCII char neighbor on either side is capitalized or not
                left = is_neighbor_capital(sampled_word, sampled_char_i, -1)
                right = is_neighbor_capital(sampled_word, sampled_char_i, 1)

                # Truth table to capitalize or not based on surrounding ASCII chars
                doCapitalize = truth_table(left, right)

                print(f"{left} - {right} | {doCapitalize}")

                # If truth table results in saying we should capitalize, then capitalize
                if doCapitalize:
                    inserted_char = random.sample(string.ascii_uppercase, 1)[0]
                else:
                    inserted_char = random.sample(string.ascii_lowercase, 1)[0]

                # Insert character and update our results
                temp = sampled_word[:sampled_char_i]
                temp += inserted_char
                temp += sampled_word[sampled_char_i:]
            if sampled_typo_type == "replacement":
                temp = list(sampled_word)
                if sampled_char.isupper():
                    replacement_char = random.sample(keyboard_info[sampled_char], 1)[0]
                else:
                    replacement_char = random.sample(keyboard_info[sampled_char.upper()][0], 1)[0].lower()
                temp[sampled_char_i] = replacement_char
                temp = "".join(temp)
            if sampled_typo_type == "deletion":
                temp = sampled_word[:sampled_char_i-1] + sampled_word[sampled_char_i:]
            if sampled_typo_type == "transposition":
                temp = sampled_word[:sampled_char_i] + sampled_word[sampled_char_i+1] + sampled_word[sampled_char_i] + sampled_word[sampled_char_i+2:]
            
            print(f"Result: {temp}")

            typo_data[data_point_i][0][sampled_word_i] = temp

    return typo_data

def list_sample(lst, samp_dist):
    '''
    Given a list lst, and a list samp_dist of the same length, choose a random item in lst based on the normalized probability inside samp_dist
    '''
    sample = random.random()
    sampled_word = None
    temp_cum = 0
    for i in range(len(samp_dist)):
        lim = samp_dist[i]
        temp_cum += lim
        if sample <= temp_cum:
            return(lst[i], i)
    return (lst[-1], len(lst)-1) # if all else fails

def is_neighbor_capital(word, index, direction):
    '''
    Finds the nearest ASCII neighbor to the left or right to the desired inserted space
    Direction is either -1 or 1 (left or right)
    If the nearest ASCII neighbor to the left is the first character, then return None
    '''
    # If inserted characters are inserted to the right of the chosen index, then the index itself is the first left neighbor 
    if direction == -1:
        diff = 0
    else:
        diff = 1
    neighbor = index + (direction * diff)

    # Loop to find the nearest neighbor, skipping non-ASCII
    while not (neighbor < 1 or neighbor > len(word)-1):
        # if neighbor == 0 or neighbor == len(word)-1:
        #     return None
        if word[neighbor] in string.ascii_letters:
            if word[neighbor] in string.ascii_uppercase:
                return True
            else:
                return False
        else:
            diff += 1
        neighbor = index + (direction * diff)
    return None

def truth_table(left, right):
    # Mimics the truth table seen in "typos_truth_table.txt"
    if left == True:
        if right == False:
            return random.choice([True, False])
        return True
    if left == False:
        if right == True:
            return random.choice([True, False])
        return False
    if right == True:
        return True
    if right == False:
        return False
    return random.choice([True, False])


In [2501]:

data_dev = read_iob2_file_wPOS(r"../data/test_conll.iob2")
x = [list(data_dev[2])]
x

[[['AL-AIN', ',', 'United', 'Arab', 'Emirates', '1996-12-06'],
  ['B-LOC', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O'],
  ['NNP', ',', 'NNP', 'NNP', 'NNPS', 'CD']]]

In [2661]:
typo_generation(x, corruption_ratio=0.05, min_typos=1, max_typos=5, corruption_variation=1)

# for _ in range(100):
#     typo_generation(x, corruption_ratio=0.05, min_typos=1, max_typos=5, corruption_variation=1)

' '.join(x[0][0])

~~~~~~~~~~~~~~~~~~~~~~~~~
Sampled word: Emraties
transposition_set_zero: [0, 2, 3, 6]
Typo type: transposition
Character sampling distribution: [0.0, 0.23999999999999996, 0.0, 0.0, 0.36, 0.39999999999999997, 0.0, 0.0]
Result: Emrateis


'LA-AIN , Utendi Arab Emrateis 1996-12-06'

In [1967]:
test = "Hewlo"
index = 2
test[:index] + test[index+1] + test[index] + test[index+2:]

'Helwo'

In [2309]:
sampled_word = "Asian"

isAllOneHand = True
transposition_set_zero = []
for i in range(len(sampled_word)-1):
    if i == 0 and sampled_word[i].isupper():
        transposition_set_zero.append(i)
        continue
    a = keyboard_info.get(sampled_word[i].upper())[1]
    b = keyboard_info.get(sampled_word[i+1].upper())[1]
    if a == None or b == None:
        transposition_set_zero.append(i)
    elif a == b:
        transposition_set_zero.append(i)
    else:
        isAllOneHand = False

print(isAllOneHand)
print(transposition_set_zero)

False
[0]
